### Ingest from our prepared dataset json

In [1]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

input_json = CWD / "result.json"
descriptions_json = CWD / "g0_descriptions.json"
if not input_json.is_file():
    raise ValueError(f"file '{input_json}' doesnt exist")
if not descriptions_json.is_file():
    raise ValueError(f"file '{descriptions_json}' doesnt exist")

# example input json
"""
 [
    { "source" : file_name, 
        "chunks": [
            { "id": hash_of_text, "raw_text" : raw_text, "approx_n_tokens": int, "embedding": list[float], 
                "triples" : [
                    { "s": subj_str, "p": pred_str, "o": object_str }, ...
                ]
            }, ...
        ] 
    }, ...
 ]
"""
print("")

In [2]:
# verify that the memgraph container is reachable
import socket

host = "localhost"
port = 7687

# DNS resolution check
socket.gethostbyname(host)

# TCP reachability check (Bolt runs over TCP)
with socket.create_connection((host, port), timeout=2):
    pass

In [3]:
import utils.mg_driver as mg_driver
await mg_driver.init()
await mg_driver.clear()

In [4]:
# upsert into memgraph
import json, ijson
from utils.models import SPOTriple, Chunk
import tqdm
from utils import normalize_from_name

with open(input_json, "rb") as in_file, open(descriptions_json, "r") as descriptions_file:
    # load description map
    data = json.load(descriptions_file)
    description_map = {}
    for obj in data:
        description_map.update(obj)
    del data

    #upsert all triples
    for itm in ijson.items(in_file, "item"):
        print(f"ingesting {itm['source']} ({len(itm['chunks'])} chunks) (id:{itm['id']})")

        for i,chunk in enumerate(itm['chunks']):
            chunk:Chunk = chunk
            for triple in chunk['triples']:
                snorm , onorm = normalize_from_name(triple['s']),normalize_from_name(triple['o'])
                pnorm = snorm + "__" + normalize_from_name(triple['p']) + "__" + onorm
                norm_triple:SPOTriple = { 's':snorm, 'p':pnorm, 'o':onorm }
                try:
                    triple_descriptions = (description_map[norm_triple['s']], description_map[norm_triple['p']], description_map[norm_triple['o']])
                except KeyError:
                    print(f"couldnt get description for a triple element! triple: {norm_triple}")
                    continue    
                await mg_driver.merge_triple(norm_triple, triple_descriptions, source_doc_id=itm['id'], source_chunk_id=chunk['id'])
            print(f"{".." if i%20!=0 else "\n.."}{i}",end="")
    

await mg_driver.close()

ingesting minecraft.pdf (59 chunks) (id:5411a9b8b270828a7771fbdb1f601355)

..0..1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..17..18..19
..20..21..22..23..24..25..26..27..28..29..30..31..32..33..34..35..36..37..38..39
..40..41..42..43..44..45..46..47..48..49..50..51..52..53..54..55..56..57..58